# FixtureDB: Mock Adoption Analysis

This notebook examines mocking patterns in test fixtures:
- **Mock adoption rate**: Percentage of fixtures using mocks by language
- **Framework diversity**: Which mocking frameworks are most popular
- **Mock styles**: Distribution of stub, mock, spy, and fake patterns

These analyses reveal how teams use mocking in fixture design.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

plt.rcParams['figure.facecolor'] = '#FAFAFA'
print("✓ Libraries imported")

In [ ]:
data_dir = Path('fixturedb')
repos = pd.read_csv(data_dir / 'repositories.csv')
fixtures = pd.read_csv(data_dir / 'fixtures.csv')
mock_usages = pd.read_csv(data_dir / 'mock_usages.csv')

print(f"Fixtures: {len(fixtures)}")
print(f"Mock usages: {len(mock_usages)}")
print(f"\nMock columns: {', '.join(mock_usages.columns.tolist())}")

## Mock Adoption Rate by Language

In [ ]:
# Calculate adoption rate
fixtures_with_lang = fixtures.merge(repos[['id', 'language']], left_on='repo_id', right_on='id', how='left')
fixtures_with_mocks = fixtures_with_lang.merge(
    mock_usages[['fixture_id']].drop_duplicates(), 
    left_on='id_x', right_on='fixture_id', how='left', indicator=True
)
fixtures_with_mocks['has_mock'] = fixtures_with_mocks['_merge'] == 'both'

adoption_by_lang = fixtures_with_mocks.groupby('language')['has_mock'].agg(['sum', 'count'])
adoption_by_lang['adoption_rate'] = adoption_by_lang['sum'] / adoption_by_lang['count']
print("Mock Adoption by Language:")
print(adoption_by_lang)

fig, ax = plt.subplots(figsize=(10, 6), facecolor='#FAFAFA')
adoption_by_lang['adoption_rate'].plot(kind='bar', ax=ax, color=sns.color_palette('husl', len(adoption_by_lang)))
ax.set_title('Mock Adoption Rate by Language', fontsize=14, fontweight='bold')
ax.set_xlabel('Language', fontsize=12)
ax.set_ylabel('Adoption Rate', fontsize=12)
ax.set_ylim([0, 1])
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(adoption_by_lang['adoption_rate']):
    ax.text(i, v + 0.02, f'{v*100:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Mock Framework Distribution

In [ ]:
if 'framework' in mock_usages.columns:
    framework_counts = mock_usages['framework'].value_counts().head(15)
    print("Top 15 Mock Frameworks:")
    print(framework_counts)
    
    fig, ax = plt.subplots(figsize=(12, 8), facecolor='#FAFAFA')
    framework_counts.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(framework_counts)))
    ax.set_title('Mock Framework Prevalence', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Usages', fontsize=12)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("'framework' column not available")

## Mock Styles (Stubs, Mocks, Spies, Fakes)

In [ ]:
if 'mock_style' in mock_usages.columns:
    style_counts = mock_usages['mock_style'].value_counts()
    print("Mock Styles:")
    print(style_counts)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#FAFAFA')
    
    # Pie chart
    colors = sns.color_palette('Set2', len(style_counts))
    ax1.pie(style_counts.values, labels=style_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
    ax1.set_title('Mock Style Distribution', fontsize=12, fontweight='bold')
    
    # Bar chart
    style_counts.plot(kind='bar', ax=ax2, color=colors)
    ax2.set_title('Mock Style Count', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Style', fontsize=11)
    ax2.set_ylabel('Count', fontsize=11)
    ax2.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
else:
    print("'mock_style' column not available in public CSV (SQLite-only)")

## Key Findings

**Mocking Patterns:**
- Mock adoption varies significantly across languages
- Popular frameworks dominate each ecosystem
- Mock style patterns reveal common testing strategies

**Note:** Some metrics (mock_style, target_layer) are qualitative and available only in the SQLite database.